In [1]:
print("hello")

hello


In [9]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# Load Variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


# TOOL
def get_weather(location: str) -> str:
    """
    Returns the weather of location
    """

    if "tokyo" in location.lower():
        return f"The weather in {location} is 72 and Sunny"
    elif "singapore" in location.lower():
        return f"The weather in {location} is 88 and humid"
    else:
        return f"The weather in {location} is 65 and cloudy"


tools = [get_weather]


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [12]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [14]:
def call_model(state: AgentState):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


tool_node = ToolNode(tools)

In [15]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", tools_condition)
workflow.add_edge("tools", "agent")

In [16]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

NameError: name 'graph' is not defined